# Task 3 — SAM with MixUp 0.2

Test whether SAM reduces the remaining gap between clean training and validation scores. SAM takes two gradient passes before one update. Keep **MixUp alpha 0.2**, **folds 0 and 4**, **30 epochs**, and **scratch training**. Only the learning step changes.

Use a **fresh Colab L4** matching the previous runs. Push this notebook and its source files first. The completed **04af** parents, earlier comparison runs and 04w precision evidence must be on Drive. The failed 04ah runs are not required.

Expect roughly twice the training compute; actual GPU time and memory will be measured. This is a development screen, not an independent blind test.

## 1. Colab GPU and repository


In [ ]:
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)


try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Commit:", commit)

## 2. Teacher data and canonical split


In [ ]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)


copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")

## 3. Freeze SAM and check the parents

The completed 04af models have a mean clean gap of **0.129133**, pooled validation F1 **0.808909**, and Unisex recall **0.510608**. Stronger MixUp 0.4 did not reduce the gap enough. A clean BatchNorm recalibration probe also failed to reduce it. SAM is a new hypothesis; its benefit here is unproven.

Use first-order, non-adaptive **L2 SAM, rho 0.05**, over the existing AdamW optimizer. For each batch:

1. Apply the existing augmentation and **MixUp alpha 0.2 once**. Compute the mixed cross-entropy gradient at the current weights.
2. Temporarily add `0.05 * gradient / (global_L2_gradient_norm + 1e-12)` to every trainable parameter, including BatchNorm affine parameters.
3. Compute a second gradient on the **same mixed batch and dropout mask**. Both passes use training batch statistics; keep running buffers and counters from the first pass only.
4. Restore the exact original weights. Apply **one AdamW update** using the second gradient. Weight decay and optimizer state advance once. Non-finite losses or gradients fail the run; exceptions restore the weights.

Keep name-truth labels, full images, widths `[32, 64, 128, 256]`, **390,181 parameters**, GeM p=3, dropout **0.30**, translation ±2 px at probability 0.50, mild darkening at probability 0.25, grayscale at probability **0.10**, clean fold-training RGB normalization, batch **128**, seed **2753**, AdamW learning rate **0.001**, weight decay **0.0001**, and cosine decay to **0.00001** over 30 epochs. No class weights, sample weights or BatchNorm recalibration. Every model starts from random weights; parents supply comparison evidence only.

Each row appears once per epoch. MixUp uses its existing dedicated PCG64 stream. Validation and held-out rows never enter either training pass. Online mixed-input F1 stays blank.

Save clean training and validation class scores at **epochs 10, 15, 20, 25 and 30**. These are diagnostics only: the **epoch-30 checkpoint** remains fixed. Extra clean checks preserve the training RNG. Final comparisons use matching name-truth labels and IEEE FP32, with the original teacher-label diagnostic retained.

The completed alpha 0.2 code hashes are checked against commit `68fef49ab1d55d671531113a71a3e71400a0e3fc`. A new source audit binds SAM code and policies. The 0.05 radius is a fixed adaptation for this trial, not a proven optimum for this AdamW model. [SAM paper](https://arxiv.org/abs/2010.01412), [official implementation](https://github.com/google-research/sam).

In [ ]:
from fashion.data.gender_name_truth import build_gender_name_truth_variant
from fashion.train.task3_gender_sam import (
    check_gender_sam_sources,
    run_gender_sam_screen,
)

G2_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender"
E6_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_e6_gem_p3/gender"
DROPOUT_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030/gender"
DARKENING_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030_mild_darkening/gender"
GRAYSCALE_DIR = (
    DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030_mild_darkening_grayscale_010/gender"
)
NAME_TRUTH_DIR = (
    DRIVE_TASK_DIR / "experiments/t3_gender_name_truth_dropout_030_grayscale_010/gender"
)
MIXUP_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_name_truth_mixup_alpha020/gender"
PRECISION_DIR = DRIVE_TASK_DIR / "diagnostics/gender_precision/20260905T085822668071Z"

summary = build_gender_name_truth_variant(REPO_DIR)
print("Changed gender labels:", summary["changed_labels"])
print("Unclear names kept:", summary["no_cue_rows"] + summary["multiple_cue_rows"])
print("Fold label changes:", summary["folds"])
sources, classes, spec, evidence = check_gender_sam_sources(
    g2_directory=G2_DIR,
    e6_directory=E6_DIR,
    dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR,
    grayscale_directory=GRAYSCALE_DIR,
    name_truth_directory=NAME_TRUTH_DIR,
    mixup_directory=MIXUP_DIR,
    source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR,
    root=REPO_DIR,
)
assert spec.classifier_dropout == 0.30
assert spec.to_dict()["grayscale_probability"] == 0.10
print("Verified source runs:", {name: len(runs) for name, runs in sources.items()})
print(
    "Direct alpha 0.2 parents:",
    {fold: run["run_id"] for fold, run in sources["MixUp20"].items()},
)
print("Frozen recipe:", spec.to_dict())
print("Output:", DRIVE_TASK_DIR / spec.artifact_dir / "gender")
print("MixUp policy:", spec.to_dict()["mixup_policy"])
print("Additional improvement rules:", spec.to_dict()["improvement_rules"])

print("SAM policy:", spec.to_dict()["sam_policy"])

## 4. Preserve the old guards and require further improvement

Keep the **14 non-F1 G2/E6 checks**, with the same labels and IEEE evaluation:

- Relative pooled, fold and class F1 changes and the paired whole-family bootstrap interval are diagnostic only. Keep 10,000 draws within folds, seed 2753; save the five replaced G2 F1 checks separately as `diagnostic_checks`.
- The mean clean gap must fall by at least 0.050 versus G2, and both folds' gaps must shrink.
- NLL may rise by at most 0.020 and ECE by at most 0.010 versus G2.
- Keep corruption guards versus matched E6: translation-induced F1 change improves by at least 0.030; each other standard corruption worsens by at most 0.020.
- Exactly 390,181 parameters and peak allocated GPU memory strictly below 3,000,000,000 bytes. Report training time and latency. A memory failure stops before another fold begins.

Add **5 checks**, using completed 04af alpha 0.2 for the gap and recall comparisons:

- Mean clean gap must shrink by **at least 0.020**: approximately **0.129133 → at most 0.109133**.
- Each fold's clean gap must shrink.
- Pooled validation macro-F1 must be **at least 0.74 (74%)**. A drop from the parent's 80.89% is allowed.
- **Unisex recall must not fall** versus 04af (approximately **0.510608**).

Use exact freshly matched parent scores, not these rounded numbers. **All 19 checks must pass.** These targets are chosen before this new fit; they do not rewrite the result of 04af. Read confidence quality and raw corruption scores as well. The 74% floor allows a validation F1 drop in exchange for a smaller clean gap.


In [ ]:
result = run_gender_sam_screen(
    g2_directory=G2_DIR,
    e6_directory=E6_DIR,
    dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR,
    grayscale_directory=GRAYSCALE_DIR,
    name_truth_directory=NAME_TRUTH_DIR,
    mixup_directory=MIXUP_DIR,
    source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
    root=REPO_DIR,
)
print("Screen:", result["status"])
print("Label basis:", result.get("comparison_label_basis"))
for row in result.get("folds", []):
    print(
        "Fold",
        row["fold"],
        "train F1:",
        row["candidate_train_f1"],
        "validation F1:",
        row["candidate_validation_f1"],
        "gap:",
        row["candidate_gap"],
    )
for gate in result.get("checks", []):
    if gate["status"] != "pass":
        print(gate)
if "reason" in result:
    print(result["reason"])
if "incremental_comparison" in result:
    incremental = result["incremental_comparison"]
    print("Direct alpha 0.2 parents:", result["direct_parent_run_ids"])
    print("F1 change versus alpha 0.2 on the same labels:", incremental["validation_delta"])
    print("Paired 95% interval:", incremental["validation_interval"])
    print("Class F1 changes:", incremental["class_f1_delta"])
    print("Induced corruption changes:", incremental["mean_induced_change_delta"])
if "incremental_comparison" in result:
    for row in result["incremental_comparison"]["folds"]:
        print("Direct-parent gap change, fold", row["fold"], ":", row["delta_gap"])

## 5. Stop and review

Stop after folds 0 and 4. Every fit is registered before its first optimizer step. Reuse requires matching code, labels, parents, recipe and training evidence. Review before any more folds, refits or held-out evaluation.

Results go to `MyDrive/MLA2/task3/experiments/t3_gender_name_truth_mixup_alpha020_sam005/gender`:

- Each run saves `mixup_training.json` and `sam_training.json`. Their row and batch counts must agree. SAM saves both loss traces, gradient norm bounds, two backward passes and one optimizer update per batch.
- `clean_epoch_diagnostics.json` saves full class scores on clean training and validation images at the five fixed epochs. `history.csv` keeps the first mixed loss, the second SAM loss and validation scores; mixed-input training F1 stays blank.
- `screen_decision.json` holds **all 19 required checks**, five diagnostic F1 checks and the historical G2/E6 status. `incremental_comparison.json` compares SAM with alpha 0.2; inherited `dropout_*` fields refer to those alpha 0.2 parents.
- `clean_gap_comparison.csv`, `ieee_oof_predictions.csv`, per-model IEEE evaluations, `label_basis_comparison.csv` and `original_label_diagnostic.json` retain predictions and label sensitivity evidence.
- `source_audit.json` and `label_variant/` bind code, sources and labels. Each run keeps its final checkpoint, hash, runtime and peak GPU memory.

Keep 04af as the current candidate unless SAM passes. Validation F1 below **74%** or a drop in Unisex recall fails this trial. Do not select an earlier checkpoint from the diagnostic scores.